In [58]:
%matplotlib inline

<div class="alert alert-info"><h4>Further reading:</h4><p>This notebook is adapted from the <a href="https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html">PyTorch: A 60 Minute Blitz</a> tutorial on the PyTorch website. For documentation and more tutorials, visit <a href="https://pytorch.org">pytorch.org</a></p></div>

# Autograd

`torch.autograd` is PyTorch’s automatic differentiation engine. This notebook will help you build a conceptual understanding of how it works.

## Summary

This notebook introduces **autograd**, PyTorch's automatic differentiation engine, which computes the gradients needed to train neural networks.

1. **Background: Forward and Backward Propagation** — a neural network is a set of nested functions defined by parameters; the forward pass runs the input through them, and the backward pass applies the chain rule from the output back to the parameters.
2. **Why do we need autograd?** — training adjusts parameters to reduce a loss; gradients tell us which direction to adjust them, and autograd computes them automatically where doing it by hand would be impractical.
3. **Differentiation using Autograd**
   - 3.1 Setting `requires_grad=True`, calling `.backward()`, and reading the result from `.grad`, checked against $\frac{dy}{dx} = 3x^2$
   - 3.2 Leaf vs. non-leaf tensors: why `y.grad` is `None`, and how `.retain_grad()` keeps an intermediate gradient
4. **Backpropagation Through Multiple Layers** — a chain of vector operations ($a \to b \to \hat{y} \to$ error), where autograd's gradients match the chain rule computed by hand
5. **What if our error function was different?** — why `.backward()` needs a scalar, and how replacing `sum` with `mean`, a weighted sum, or squared error changes the gradients


## 1. Background: Forward and Backward Propagation
Neural networks (NNs) are collections of nested functions that are executed on some input data. These functions are defined by *parameters* (consisting of weights and biases), which in PyTorch are stored in tensors (see the previous notebook for more on tensors).

Training a NN happens in two steps:

**Forward Propagation**: In a nutshell, the nested functions (the NN) take an input (such as an image), pass it through the functions, and give an output (for example, a classification of the image as a cat or dog).

**Backward Propagation**: In backprop, the NN computes gradients so we can update parameters proportionate to the gradient of the loss in its output. It does this by traversing backwards, starting from the output and moving toward the input, collecting the derivatives of the error with respect to the parameters of the functions (*gradients*), and using an optimizer to optimize the parameters using gradient descent. For a more detailed walkthrough of backprop, check out [this video](https://www.youtube.com/watch?v=tIeHLnjs5U8) from Grant Sanderson (3Blue1Brown).

## 2. Why do we need autograd?

Training a neural network involves adjusting its parameters (weights and biases) to reduce its prediction error, measured by a **loss function**.

To decide how to adjust each parameter, we calculate the derivative of the loss with respect to that parameter. These derivatives, collectively called **gradients**, tell us how small changes in the parameters affect the loss.

For a simple function such as $y = x^3$, we can easily calculate the derivative by hand: $\frac{dy}{dx} = 3x^2$. But neural networks involve many nested operations, so calculating all these derivatives quickly becomes difficult.

This is why autograd is the key to training neural networks: it computes these derivatives automatically, even when the calculation involves many steps. It keeps track of the operations performed in the forward pass, then runs backpropagation through them to get the gradients. An **optimizer** then uses the gradients to update the parameters and reduce the loss.

In [59]:
import torch

## 3. Differentiation using Autograd

### 3.1 A Scalar Example: $y = x^3$

Let's take a look at how ``autograd`` collects gradients with a very simple example. We'll create a one-dimensional tensor ``x`` with shape ``(1,)`` (a vector containing one element) and set ``requires_grad=True``. This signals to ``autograd`` that every operation on ``x`` should be tracked.

In [60]:
x = torch.tensor([5.], requires_grad=True)
print(x)

tensor([5.], requires_grad=True)


Now let's make another tensor, ``y``, that's a function of ``x``:

$$y = x^3$$

In [61]:
y = x ** 3
print(y)

# x.grad is empty because we haven't called backward() yet.
# y.grad is empty for a different reason — see the note below!
print(y.grad)
print(x.grad)

tensor([125.], grad_fn=<PowBackward0>)
None
None


/var/folders/ys/4vmt6skn47v69q7dptm_7p3m0000gp/T/ipykernel_5016/3365296921.py:6: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:493.)
  print(y.grad)


<div class="alert alert-warning"><h4>About that warning: leaf vs. non-leaf tensors</h4>

Printing `y.grad` triggers a `UserWarning`. Don't ignore it — it's telling you something important about how autograd stores gradients.

PyTorch divides tensors into two kinds:

- **Leaf tensors** are the ones *you* create directly, like `x = torch.tensor([5.], requires_grad=True)`. In a real network, these are the parameters (weights and biases) you want to update.
- **Non-leaf tensors** are produced *by operations* on other tensors, like `y = x ** 3`. They sit in the middle of the computation graph. You can spot them by their `grad_fn` attribute (`y` has `grad_fn=<PowBackward0>`, `x` has none).

Backprop computes gradients for non-leaf tensors as it goes — it has to, since the chain rule passes through them — but it **discards them immediately** after using them, to save memory. Only leaf tensors keep their gradient in `.grad`.

So `y.grad` is `None` *before* `backward()`, and it will still be `None` *after* `backward()`. That's not a bug; PyTorch is warning you because most people who reach for `y.grad` actually want the leaf tensor's gradient, `x.grad`.
</div>

Notice that ``y`` has a ``grad_fn`` attribute. The gradient here is just a derivative:

$$ \frac{\partial y}{\partial x} = 3x^2 $$

And we can check if autograd did its job correctly by calling ``.backward()`` on ``y``, which will store the gradient in x.grad. We expect that to be the same as $3x^2$.

In [62]:
y.backward() #this is the function that takes gradients. It automatically computes the gradients of y with respect to x and stores them in x.grad.

if x.grad == 3 * x ** 2:
    print("Gradients match!")
    print(x.grad)

Gradients match!
tensor([75.])


### 3.2 Intermediate Gradients with `.retain_grad()`

Let's confirm the claim from the note above: now that `backward()` has run, is `y.grad` populated? And what do we do if we actually *want* an intermediate gradient?

In [63]:
# x.grad was filled in, but y.grad is still empty — even after backward().
print("x.grad:", x.grad)
print("y.grad:", y.grad)   # None, and PyTorch warns again

# If we genuinely want an intermediate tensor's gradient, we ask for it
# with .retain_grad() *before* calling backward().
x2 = torch.tensor([5.], requires_grad=True)
y2 = x2 ** 3
y2.retain_grad()           # "please keep this one"
z2 = y2 * 2                # z = 2y = 2x^3

z2.backward()
print("\nWith retain_grad():")
print("y2.grad:", y2.grad)  # dz/dy = 2
print("x2.grad:", x2.grad)  # dz/dx = 2 * 3x^2 = 150

x.grad: tensor([75.])
y.grad: None

With retain_grad():
y2.grad: tensor([2.])
x2.grad: tensor([150.])


/var/folders/ys/4vmt6skn47v69q7dptm_7p3m0000gp/T/ipykernel_5016/386728996.py:3: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:493.)
  print("y.grad:", y.grad)   # None, and PyTorch warns again


**Takeaway:** the warning is a reminder to ask *which* tensor you actually want the gradient of.

Keep an eye out for `b.retain_grad()` in the next example.

------

## 4. Backpropagation Through Multiple Layers

Now let's do a slightly more complex example: Let's say we have a tensor ``a``, which represents the second-to-last hidden layer of a neural net, ``b``, which represents the last hidden layer, and ``y_hat``, which represents the output. These will all be one-dimensional tensors with shape ``(3,)``: vectors containing three elements, rather than the one-element vector in Section 3. 

![example2_1](example2_1.png)

In [64]:
a = torch.tensor([1., 2., 3.], requires_grad=True)
b = a ** 3
b.retain_grad()
y_hat = b ** 2

In [65]:
print(y_hat)

tensor([  1.,  64., 729.], grad_fn=<PowBackward0>)


And we'll say for simplicity that the error is just the sum of the elements of $\hat{y}$.

In [66]:
error = torch.sum(y_hat)

After calling ``.backward()`` on ``error``, its gradients with respect to ``a`` and ``b`` should be stored in ``a.grad`` and ``b.grad``. Are they what we'd expect?

For each component $i$, $b_i = a_i^3$ and $\hat{y}_i = b_i^2$. Each output depends only on the corresponding input component, so

$$
\begin{align}
\frac{\partial \hat{y}_i}{\partial b_i} &= 2b_i \\
\frac{\partial \hat{y}_i}{\partial a_i} &= \frac{\partial \hat{y}_i}{\partial b_i} \frac{\partial b_i}{\partial a_i} = (2b_i)(3a_i^2) = 6b_i a_i^2.
\end{align}
$$

Since $E = \text{error} = \sum_j \hat{y}_j$, we have $\partial E / \partial \hat{y}_i = 1$. Therefore, the gradients stored by autograd are

$$
\frac{\partial E}{\partial b_i} = 2b_i,
\qquad
\frac{\partial E}{\partial a_i} = 6b_i a_i^2.
$$

In [67]:
error.backward()
if all(a.grad == 6 * b * a ** 2) and all(b.grad == 2 * b):
    print("Gradients match!")

Gradients match!


------

## 5. What if our error function was different?

Calling .backward() without arguments requires a tensor containing just one number. Since y_hat contains three numbers, we use torch.sum(y_hat) to combine them into a single error value. We can then call error.backward() to compute the gradients of that error. 
Summing is one simple way to do this.

Changing the error function changes its derivative with respect to each output, which changes the gradients propagated back to b and a:

| Error | $\frac{\partial\,\text{error}}{\partial \hat{y}}$ | `b.grad` | `a.grad` |
|---|---|---|---|
| `torch.sum(y_hat)` | `1` | `2*b` | `6*b*a**2` |
| `torch.mean(y_hat)` | `1/3` | `2*b/3` | `2*b*a**2` |
| `torch.sum(w * y_hat)` (weights `w`) | `w` | `w*2*b` | `w*6*b*a**2` |
| `torch.sum((y_hat - y)**2)` (target `y`, squared error) | `2*(y_hat - y)` | `2*(y_hat-y)*2*b` | `2*(y_hat-y)*6*b*a**2` |

Squared error is one loss used to train networks: the error measures how far the output is from a **target** `y`, and the gradient shrinks to 0 as `y_hat` gets close to `y`.

Let's check the squared-error row. We rebuild `a`, `b`, and `y_hat` from scratch, because `.grad` **adds up** across calls to `.backward()`. Reusing the old `a` would add the new gradients on top of the ones from Section 4.

In [68]:
a = torch.tensor([1., 2., 3.], requires_grad=True)
b = a ** 3
b.retain_grad()
y_hat = b ** 2

y = torch.tensor([0., 60., 700.])   # a made-up target
error = torch.sum((y_hat - y) ** 2)
error.backward()

dE_dyhat = 2 * (y_hat - y)           # the new first factor in the chain rule
print("dError/dy_hat:", dE_dyhat)
print("b.grad matches:", torch.allclose(b.grad, dE_dyhat * 2 * b))
print("a.grad matches:", torch.allclose(a.grad, dE_dyhat * 6 * b * a ** 2))

#torch.allclose(a, b) checks whether all corresponding values in two tensors are approximately equal. It returns True or False.
#Unlike ==, it allows small differences caused by floating-point rounding.

dError/dy_hat: tensor([ 2.,  8., 58.], grad_fn=<MulBackward0>)
b.grad matches: True
a.grad matches: True
